In [ ]:
# Требуется, чтобы локаль поддерживала русский язык
import cbrapi as cb
import pandas as pd

from function.api_in_function import get_deposit_rates

In [ ]:
key = cb.get_key_rate(first_date='2022-01-01',period='M').reset_index()
key

In [ ]:
dep_rate = get_deposit_rates(start_years=2022)
dep_rate['date'] = dep_rate['date'].dt.to_period('M')
dep_rate

In [ ]:
dep_rate['rate'] = dep_rate['rate'].shift(periods=-1)
dep_rate = dep_rate.dropna(subset=['rate']).reset_index(drop=True)
dep_rate

In [ ]:
result = pd.merge(
    key,
    dep_rate,
    left_on='DATE',
    right_on='date',
    how= 'left',
)

result

In [ ]:
result = result.iloc[1:].reset_index(drop=True)
result = result[[
    'DATE',
    'KEY_RATE',
    'rate'
]]

result['rate'] = result['rate'].ffill()
result['DATE'] = result['DATE'].dt.to_timestamp()
result

In [ ]:
avg = result['KEY_RATE'].mean() - result['rate'].mean()
avg

In [ ]:
import plotly.express as px
import plotly.graph_objects as go

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=result['DATE'], y=result['KEY_RATE'],
    mode='lines+markers',
    name='Ключевая ставка ЦБ, %',
))
fig.add_trace(go.Scatter(
    x=result['DATE'], y=result['rate'],
    mode='lines+markers',
    name='Ставка депозита, %',
))
fig.update_layout(
    title='Динамика ставок (2025–2026)',
    xaxis_title='Дата',
    yaxis_title='Ставка, %',
    hovermode='x unified',
    legend={'orientation': 'h', 'yanchor': 'bottom', 'y': 1.02, 'xanchor': 'right', 'x': 1},
)

fig.add_annotation(
    x=0.4, y=0.2,           # 2% от левого края, 98% от низа
    xref='paper', yref='paper',
    text=f"Средняя разница между ставками: {avg:.2f}%",
    showarrow=False,
    font={'size': 14},
    bgcolor='rgba(255,255,255,0.8)',
    bordercolor='gray',
    borderwidth=1,
    xanchor='left',
    yanchor='top',
)
fig.show()

In [ ]:
# mask_ofz_in = svodnay_dont_merge['Инструмент'] == 'ОФЗ ИН (л)'

# # Очистка инфляции: NaN для ОФЗ ИН, расчёт для остальных
# svodnay_dont_merge['Очистка инфляции'] = np.where(
#     mask_ofz_in,
#     np.nan,
#     svodnay_dont_merge['Итоговая сумма'] / float(inf_factor)
# )

# # Реальный доход: для ОФЗ ИН — без очистки, для остальных — с очисткой
# svodnay_dont_merge['Реальный доход'] = np.where(
#     mask_ofz_in,
#     svodnay_dont_merge['Итоговая сумма'] - svodnay_dont_merge['На руках у человека'],
#     svodnay_dont_merge['Очистка инфляции'] - svodnay_dont_merge['На руках у человека']
# )
